# Practical 8: Fine-Tuning and Deployment of Open-Source Generative AI Models

- **Name:** Vishal Bhutekar
- **Roll No:** BT24S05F002


In [8]:

!pip install -q transformers datasets accelerate torch

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    pipeline
)

# 1. Select device
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# 2. Load an open-source pretrained model
model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)

# 3. Prepare a small training dataset
texts = [
    "Generative AI creates new content such as text, images, and music.",
    "Machine learning enables computers to learn patterns from data.",
    "Hugging Face provides tools for training and deploying AI models.",
    "Deep learning uses neural networks to solve complex problems.",
    "Natural language processing helps computers understand human language.",
    "Transformers are widely used in modern generative AI applications.",
    "Fine-tuning adapts a pretrained model to a specific task.",
    "Python is a popular programming language for artificial intelligence."
] * 10

dataset = Dataset.from_dict({"text": texts})

# 4. Tokenize the dataset
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=64
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

# 5. Configure training
training_args = TrainingArguments(
    output_dir="./generative_ai_model",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    save_strategy="no",
    logging_steps=10,
    report_to="none",
    use_cpu=(device == "cpu")
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# 6. Fine-tune the model
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

print("\nStarting model fine-tuning...")
trainer.train()

# 7. Save the fine-tuned model
save_path = "./fine_tuned_generative_model"

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print("\nModel fine-tuned and saved successfully!")

# 8. Deploy locally using a text-generation pipeline
generator = pipeline(
    "text-generation",
    model=save_path,
    tokenizer=save_path,
    device=0 if device == "cuda" else -1
)

# 9. Generate text using the fine-tuned model
prompt = "Generative artificial intelligence"

result = generator(
    prompt,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.8,
    pad_token_id=tokenizer.eos_token_id
)

print("\nGenerated Text:")
print(result[0]["generated_text"])

print("\nDeployment completed: the model is ready for local text generation.")


Using device: cpu


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model fine-tuned and saved successfully!


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'pad_token_id', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



Generated Text:
Generative artificial intelligence enables computers to learn patterns from data. Artificial intelligence enables computers to learn patterns from data.

































Deployment completed: the model is ready for local text generation.
